In [14]:
import pandas as pd

# CSV-Datei einlesen
# Quelle: https://www.bbsr.bund.de/BBSR/DE/forschung/raumbeobachtung/Raumabgrenzungen/downloads/download-referenzen.html
df = pd.read_excel("../data/raw/raumgliederungen-referenzen-2023.xlsx", skiprows=1, sheet_name="Kreisreferenz")

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 66 columns):
 #   Column                                                               Non-Null Count  Dtype 
---  ------                                                               --------------  ----- 
 0   Kreise (2023) Kennziffer                                             400 non-null    int64 
 1   Kreise (2023) Name                                                   400 non-null    object
 2   Arbeitsagenturbezirke (2023) Kennziffer                              400 non-null    int64 
 3   Arbeitsagenturbezirke (2023) Name                                    400 non-null    object
 4   Arbeitsmarktregionen (2023) Kennziffer                               400 non-null    int64 
 5   Arbeitsmarktregionen (2023) Name                                     400 non-null    object
 6   Braunkohlereviere (auch nicht förderfähige) (2023) Kennziffer        400 non-null    int64 
 7   Braunkohlereviere

In [15]:
df=df[["Kreise (2023) Name", "Raumordnungsregionen (2023) Name"]]
df=df.rename(columns={"Kreise (2023) Name": "Name", "Raumordnungsregionen (2023) Name": "ROR"})
df2 = pd.read_csv("../data/processed/hilfen_2024.csv")

import re
names = (
    df2["Name"]
    .dropna()
    .astype(str)
    .str.lower()
    .apply(re.escape)  
)

pattern = r"\b(" + "|".join(names) + r")\b"

df = df.loc[
    df["Name"]
      .astype(str)
      .str.lower()
      .str.contains(pattern, na=False, regex=True)
]

assert(len(df) == 52)



C:\Users\inaba\AppData\Local\Temp\ipykernel_3416\3998078846.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, na=False, regex=True)


In [16]:
from ipynb.fs.defs.functions import clean_and_sort, validate_df
df = df[df["Name"] != "Essen, krfr. Stadt"]

df = clean_and_sort(df, "ROR")

In [17]:
#df = df.merge(df2, on=["ROR"], how="left")
set_hilfen = set(df2["Name"])
set_ror = set(df["Name"])
no_match = set_ror.symmetric_difference(set_hilfen)
print(no_match)

{'Hagen, Stadt der FernUniversität', 'Hagen', 'Solingen, Klingenstadt', 'Solingen'}


In [19]:
mapping = {'Hagen, Stadt der FernUniversität':'Hagen', 'Solingen, Klingenstadt': 'Solingen'}
df["Name"] = df["Name"].replace(mapping)

print(df, df.info())
df.to_csv("../data/processed/ror.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Name    52 non-null     object
 1   ROR     52 non-null     object
dtypes: object(2)
memory usage: 964.0+ bytes
                          Name             ROR
0                   Düsseldorf      Düsseldorf
1                     Duisburg  Duisburg/Essen
2                      Krefeld      Düsseldorf
3              Mönchengladbach      Düsseldorf
4          Mülheim an der Ruhr  Duisburg/Essen
5                   Oberhausen  Duisburg/Essen
6                    Remscheid      Düsseldorf
7                     Solingen      Düsseldorf
8                    Wuppertal      Düsseldorf
9                        Kleve  Duisburg/Essen
10                    Mettmann      Düsseldorf
11           Rhein-Kreis Neuss      Düsseldorf
12                     Viersen      Düsseldorf
13                       Wesel  Duisburg/Essen
14  